# Pricing and Hedging of European Options

Cox-Ross-Rubinstein trees, Monte-Carlo simulation, the closed-form Black-Scholes formula, and three finite-difference schemes for the Black-Scholes PDE.

## Contents
1. [Setup](#1-setup)
2. [Risk-neutral probability](#2-risk-neutral-probability)
3. [Direct pricer](#3-direct-pricer)
4. [Backward-induction pricer](#4-backward-induction-pricer)
5. [Dynamic hedging](#5-dynamic-hedging)
6. [Monte-Carlo pricer](#6-monte-carlo-pricer)
7. [Closed-form Black-Scholes put](#7-closed-form-black-scholes-put)
8. [Monte-Carlo vs closed form](#8-monte-carlo-vs-closed-form)
9. [CRR convergence to Black-Scholes](#9-crr-convergence-to-black-scholes)
10. [Black-Scholes PDE: finite differences](#10-black-scholes-pde-finite-differences)
11. [Choosing M](#11-choosing-m)


## 1. Setup

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from math import comb
from scipy.stats import norm


## 2. Risk-neutral probability

Discrete grid $t_0=0<\dots<t_N=T$, $\delta=T/N$. Riskless asset $(1+r_N)^i$; risky asset $S^{(N)}_{t_i}=T^{(N)}_i S^{(N)}_{t_{i-1}}$ with $T^{(N)}_i\in\{1+h_N,\,1+b_N\}$ i.i.d., $b_N<r_N<h_N$.

$Q$ makes the discounted price a martingale:

$$q_N = \frac{r_N - b_N}{h_N - b_N}$$

Summing over the number of up-moves $k$:

$$Prix^{(N)} = \frac{1}{(1+r_N)^N}\sum_{k=0}^{N} \binom{N}{k} q_N^k (1-q_N)^{N-k}\, f\!\left(s(1+h_N)^k(1+b_N)^{N-k}\right)$$


## 3. Direct pricer

Closed-form binomial sum.

In [ ]:
def price1(N, rN, hN, bN, s, f):
    """Binomial price via direct summation over the number of up-moves."""
    qN = (rN - bN) / (hN - bN)
    total = 0.0
    for k in range(N + 1):
        ST = s * (1 + hN)**k * (1 + bN)**(N - k)
        proba = math.comb(N, k) * qN**k * (1 - qN)**(N - k)
        total += proba * f(ST)
    return total / (1 + rN)**N


Test.

In [ ]:
call = lambda x: max(x - 100, 0)
price1(20, 0.01, 0.05, -0.05, 100, call)


## 4. Backward-induction pricer

$$v_N(S_{t_N}) = f(S_{t_N}), \qquad v_k(S_{t_k}) = \frac{1}{1+r_N}\Big(q_N\, v_{k+1}(\text{up}) + (1-q_N)\, v_{k+1}(\text{down})\Big)$$

In [ ]:
def price2(N, rN, hN, bN, s, f):
    """Binomial price via backward induction on the recombining tree."""
    qN = (rN - bN) / (hN - bN)
    prices = np.zeros(N + 1)
    for k in range(N + 1):
        prices[k] = s * (1 + hN)**k * (1 + bN)**(N - k)
    V = [f(x) for x in prices]
    for k in range(N - 1, -1, -1):
        for i in range(k + 1):
            V[i] = (1 / (1 + rN)) * (qN * V[i + 1] + (1 - qN) * V[i])
    return V


Test.

In [ ]:
call = lambda x: max(x - 100, 0)
price2(3, 0.01, 0.05, -0.05, 100, call)[0]


## 5. Dynamic hedging

At date $t_{k-1}$ with asset price $x$, matching $v_k$ in both states gives

$$\alpha_{k-1}(x) = \frac{v_k\big((1+h_N)x\big) - v_k\big((1+b_N)x\big)}{x(h_N - b_N)}, \qquad \beta_{k-1}(x) = \frac{(1+h_N)\,v_k\big((1+b_N)x\big) - (1+b_N)\,v_k\big((1+h_N)x\big)}{(1+r_N)^k(h_N - b_N)}$$

In [ ]:
def hedge(N, rN, hN, bN, s, f):
    """Replicating strategy (alpha, beta) at every node before maturity."""
    alpha = []
    beta = []
    for k in range(1, N + 1):
        alpha_k = []
        beta_k = []
        for j in range(k):
            x = s * (1 + hN)**j * (1 + bN)**(k - 1 - j)
            v_up = price2(N - k, rN, hN, bN, (1 + hN) * x, f)[0]
            v_down = price2(N - k, rN, hN, bN, (1 + bN) * x, f)[0]
            a = (v_up - v_down) / (x * (hN - bN))
            b = ((1 + hN) * v_down - (1 + bN) * v_up) / ((1 + rN)**k * (hN - bN))
            alpha_k.append(a)
            beta_k.append(b)
        alpha.append(alpha_k)
        beta.append(beta_k)
    return alpha, beta


Test.

In [ ]:
call = lambda x: max(x - 100, 0)
hedge(2, 0.01, 0.05, -0.05, 100, call)


## 6. Monte-Carlo pricer

$$S_t = s\exp\!\left[\left(r - \frac{\sigma^2}{2}\right)t + \sigma B_t\right], \qquad \widehat{P}^{(n)} = \frac{1}{n}\sum_{i=1}^n e^{-rT} f\!\left(s\exp\!\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,\xi_i\right]\right)$$

The summands are i.i.d. with mean $P$, so $\widehat{P}^{(n)} \to P$ almost surely by the strong law of large numbers.

In [ ]:
def price3(n, s, r, sigma, T, f):
    """Monte-Carlo price of an option paying f(S_T)."""
    xi = np.random.randn(n)
    ST = s * np.exp((r - sigma**2 / 2) * T + sigma * np.sqrt(T) * xi)
    payoffs = np.array([f(v) for v in ST])
    return np.exp(-r * T) * np.mean(payoffs)


## 7. Closed-form Black-Scholes put

$$Prix_{BS} = -sF(-d) + Ke^{-rT}F(-d+\sigma\sqrt{T}), \qquad d = \frac{1}{\sigma\sqrt{T}}\left[\ln\!\left(\frac{s}{K}\right) + \left(r+\frac{\sigma^2}{2}\right)T\right]$$

In [ ]:
def put(s, r, sigma, T, K):
    """Closed-form Black-Scholes put price. Vectorised over s."""
    d = (np.log(s / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
    return -s * norm.cdf(-d) + K * np.exp(-r * T) * norm.cdf(-d + sigma * np.sqrt(T))


Test.

In [ ]:
put(100, 0.01, 0.1, 1, 90)


## 8. Monte-Carlo vs closed form

Monte-Carlo price against the number of simulations, with the closed-form value as reference.

In [ ]:
r = 0.01
sigma = 0.1
s = 100
T = 1
f = lambda x: max(100 - x, 0)

n_values = [10**5 * k for k in range(1, 11)]
mc_prices = [price3(n, s, r, sigma, T, f) for n in n_values]
bs_price = put(s, r, sigma, T, K=100)


Plot.

In [ ]:
plt.plot(n_values, mc_prices, marker='o', label='Monte-Carlo')
plt.axhline(y=bs_price, color='red', label='Black-Scholes closed form')
plt.xlabel('Number of simulations (n)')
plt.ylabel('Option price')
plt.legend()
plt.show()


Relative error, log scale.

In [ ]:
errors = [abs(p - bs_price) / abs(bs_price) for p in mc_prices]

plt.plot(n_values, errors, marker='o')
plt.xlabel('Number of simulations (n)')
plt.ylabel('Relative error')
plt.yscale('log')
plt.show()


**Observation.** The Monte-Carlo estimate oscillates around the closed-form price without converging monotonically: each $\widehat{P}^{(n)}$ is itself a random variable, so a larger $n$ can land further from $P$ than a smaller one. The strong law guarantees convergence as $n\to\infty$, and the central limit theorem puts the typical error at order $1/\sqrt{n}$. A monotone decay would only appear by averaging several independent runs per $n$.

## 9. CRR convergence to Black-Scholes

With $r_N = rT/N$, $h_N = (1+r_N)e^{\sigma\sqrt{T/N}} - 1$, $b_N = (1+r_N)e^{-\sigma\sqrt{T/N}} - 1$.

In [ ]:
r = 0.03
sigma = 0.3
s = 100
T = 1
f = lambda x: max(100 - x, 0)

N_values = [10 * k for k in range(1, 101)]
crr_prices = []
for N in N_values:
    rN = r * T / N
    hN = (1 + rN) * math.exp(sigma * math.sqrt(T / N)) - 1
    bN = (1 + rN) * math.exp(-sigma * math.sqrt(T / N)) - 1
    crr_prices.append(price2(N, rN, hN, bN, s, f)[0])

bs_price = put(s, r, sigma, T, K=100)


Plot.

In [ ]:
plt.plot(N_values, crr_prices, label='CRR')
plt.axhline(y=bs_price, color='red', label='Black-Scholes closed form')
plt.xlabel('Number of discretization steps (N)')
plt.ylabel('Option price')
plt.legend()
plt.show()


**Observation.** The CRR price converges to the Black-Scholes price as $N$ grows. Unlike Monte-Carlo, the convergence is deterministic: the gap comes purely from time discretization, not sampling noise. The curve overshoots for small $N$ and settles into damped oscillations that narrow steadily.

## 10. Black-Scholes PDE: finite differences

$$\frac{\partial p}{\partial t} - \frac{\sigma^2}{2}\frac{\partial^2 p}{\partial x^2} - \left(r-\frac{\sigma^2}{2}\right)\frac{\partial p}{\partial x} + rp = 0, \qquad x = \ln(S)$$

with $p(0,x) = \max(K-e^x,0)$, $p(t,x_{min}) = Ke^{-rt}-e^{x_{min}}$, $p(t,x_{max}) = 0$.

Grid: $\Delta t = T/M$, $h = (x_{max}-x_{min})/N$. Writing $A = \dfrac{\sigma^2\Delta t}{2h^2}$ and $B = \left(r-\dfrac{\sigma^2}{2}\right)\dfrac{\Delta t}{2h}$, the three schemes share the coefficients

$$a = A-B, \qquad b = 1-2A-r\Delta t, \qquad c = A+B$$

- **Explicit**: $p_{m+1,j} = a\,p_{m,j-1} + b\,p_{m,j} + c\,p_{m,j+1}$ — direct, but only stable under a CFL condition.
- **Implicit**: $-a\,p_{m+1,j-1} + (2-b)\,p_{m+1,j} - c\,p_{m+1,j+1} = p_{m,j}$ — one tridiagonal solve per step, unconditionally stable.
- **Crank-Nicolson**: the average of the two, $O(\Delta t^2)$ in time, unconditionally stable.

### Explicit scheme

In [ ]:
def solveEDP_explicit(K, r, sigma, T, M, N, xmax, xmin):
    delta_t = T / M
    t = np.array([m * delta_t for m in range(M + 1)])
    h = (xmax - xmin) / N
    x = np.array([xmin + j * h for j in range(N + 1)])
    p = np.zeros((M + 1, N + 1))

    p[0, :] = np.maximum(K - np.exp(x), 0)
    p[:, 0] = K * np.exp(-r * t) - np.exp(xmin)
    p[:, N] = 0

    diffusion = sigma**2 * delta_t / (2 * h**2)
    drift = (r - sigma**2 / 2) * delta_t / (2 * h)
    a = diffusion - drift
    b = 1 - 2 * diffusion - r * delta_t
    c = diffusion + drift

    for m in range(0, M):
        for j in range(1, N):
            p[m + 1][j] = a * p[m][j - 1] + b * p[m][j] + c * p[m][j + 1]

    return p[M, :]


### Implicit scheme

In [ ]:
def solveEDP_implicit(K, r, sigma, T, M, N, xmax, xmin):
    delta_t = T / M
    t = np.array([m * delta_t for m in range(M + 1)])
    h = (xmax - xmin) / N
    x = np.array([xmin + j * h for j in range(N + 1)])
    p = np.zeros((M + 1, N + 1))

    p[0, :] = np.maximum(K - np.exp(x), 0)
    p[:, 0] = K * np.exp(-r * t) - np.exp(xmin)
    p[:, N] = 0

    diffusion = sigma**2 * delta_t / (2 * h**2)
    drift = (r - sigma**2 / 2) * delta_t / (2 * h)
    alpha = drift - diffusion
    beta = 1 + 2 * diffusion + r * delta_t
    gamma = -diffusion - drift

    n = N - 1
    A = beta * np.eye(n) + gamma * np.eye(n, k=1) + alpha * np.eye(n, k=-1)

    for m in range(M):
        b = np.zeros(n)
        b[0] = -alpha * p[m + 1, 0]
        b[-1] = -gamma * p[m + 1, N]
        p[m + 1, 1:N] = np.linalg.solve(A, p[m, 1:N] + b)

    return p[M, :]


### Crank-Nicolson scheme

In [ ]:
def solveEDP_CN(K, r, sigma, T, M, N, xmax, xmin):
    delta_t = T / M
    t = np.array([m * delta_t for m in range(M + 1)])
    h = (xmax - xmin) / N
    x = np.array([xmin + j * h for j in range(N + 1)])
    p = np.zeros((M + 1, N + 1))

    p[0, :] = np.maximum(K - np.exp(x), 0)
    p[:, 0] = K * np.exp(-r * t) - np.exp(xmin)
    p[:, N] = 0

    diffusion = sigma**2 * delta_t / (2 * h**2)
    drift = (r - sigma**2 / 2) * delta_t / (2 * h)
    a = diffusion - drift
    b = 1 - 2 * diffusion - r * delta_t
    c = diffusion + drift

    n = N - 1
    C_diag, C_sub, C_super = (1 + b) / 2, a / 2, c / 2
    B_diag, B_sub, B_super = 1 - (b - 1) / 2, -a / 2, -c / 2

    C = C_diag * np.eye(n) + C_super * np.eye(n, k=1) + C_sub * np.eye(n, k=-1)
    B = B_diag * np.eye(n) + B_super * np.eye(n, k=1) + B_sub * np.eye(n, k=-1)

    for m in range(M):
        rhs = C @ p[m, 1:N]
        rhs[0] += C_sub * p[m, 0] - B_sub * p[m + 1, 0]
        rhs[-1] += C_super * p[m, N] - B_super * p[m + 1, N]
        p[m + 1, 1:N] = np.linalg.solve(B, rhs)

    return p[M, :]


### Running the three schemes

In [ ]:
K = 1
r = 0.015
sigma = 0.21
T = 1
N = 100
xmin = math.log(0.4)
xmax = math.log(2)
M = 500

h = (xmax - xmin) / N
x = np.array([xmin + j * h for j in range(N + 1)])

pM_explicit = solveEDP_explicit(K, r, sigma, T, M, N, xmax, xmin)
pM_implicit = solveEDP_implicit(K, r, sigma, T, M, N, xmax, xmin)
pM_cn = solveEDP_CN(K, r, sigma, T, M, N, xmax, xmin)

exact = put(np.exp(x), r, sigma, T, K)


### Price curves

In [ ]:
plt.plot(x, pM_explicit, label='Explicit')
plt.plot(x, pM_implicit, label='Implicit')
plt.plot(x, pM_cn, label='Crank-Nicolson')
plt.plot(x, exact, '--', label='Black-Scholes closed form')
plt.xlabel('x = ln(S)')
plt.ylabel('Option price')
plt.legend()
plt.show()


### Relative errors

In [ ]:
err_explicit = np.abs(pM_explicit - exact) / np.abs(exact)
err_implicit = np.abs(pM_implicit - exact) / np.abs(exact)
err_cn = np.abs(pM_cn - exact) / np.abs(exact)

plt.plot(x, err_explicit, label='Explicit')
plt.plot(x, err_implicit, label='Implicit')
plt.plot(x, err_cn, label='Crank-Nicolson')
plt.yscale('log')
plt.xlabel('x = ln(S)')
plt.ylabel('Relative error')
plt.legend()
plt.show()


**Observation.** All three schemes match the closed-form price closely over most of the domain (relative errors around $10^{-4}$ to $10^{-6}$ in the central region). The error grows sharply near $x_{max}$, where the put price tends to zero and dividing by it inflates the relative error — an artefact of the metric, not of the schemes. With a fine time step the theoretical gap between $O(\Delta t)$ and $O(\Delta t^2)$ accuracy is not visible.

## 11. Choosing M

The explicit scheme is only stable under a CFL condition linking $\Delta t$ to $h^2$. With the parameters above:

$$h = \frac{\ln 2 - \ln 0.4}{100} \approx 0.0161 \;\Rightarrow\; h^2 \approx 2.59\times 10^{-4}, \qquad \sigma^2 = 0.0441$$

$$\Delta t \le \frac{h^2}{\sigma^2} \approx 5.87\times 10^{-3} \;\Rightarrow\; M \ge \frac{T}{\Delta t} \approx 170$$

| $M$ | Explicit | Implicit | Crank-Nicolson |
|---|---|---|---|
| 50 | unstable, diverges | smooth, visible error | smooth, small error |
| 300 | stable | stable | stable |
| 500 | stable, comfortable margin | stable | stable, smallest error |

$M = 500$ is used above: it clears the stability threshold with margin, keeps all three curves superposed on the price plot, and leaves Crank-Nicolson with the smallest relative error, followed by the implicit and explicit schemes.